# 🎬 ComfyUI + Wan2.2 (Image-to-Video) — Google Colab ワンクリック起動

[Zuntan03/EasyWan22](https://github.com/Zuntan03/EasyWan22) の構成を参考に、Google Colab 上で
**ComfyUI + ComfyUI-Manager + Wan2.2 (I2V) 関連カスタムノード + モデル一式** を自動セットアップし、
Cloudflare Tunnel 経由でブラウザから ComfyUI を操作できるようにするノートブックです。

## 使い方（生徒向け）
1. メニューの **「ランタイム」→「ランタイムのタイプを変更」** で **GPU（T4など）** を選択します。
2. メニューの **「ランタイム」→「すべてのセルを実行」** をクリックします（上から順に自動実行されます）。
3. 一番下のセルの出力に表示される **`https://xxxxx.trycloudflare.com`** のURLをクリックすると、ブラウザでComfyUIが開きます。
4. ComfyUI が開いたら、右上メニューの **「Workflow」→「Browse Templates」→「Video」** から **Wan2.2 Image to Video** のテンプレートを選ぶと、ダウンロード済みのモデルが自動で選択されたワークフローが読み込まれます。
5. （オプション）Wan2.2は映像のみで音声は生成されません。動画に効果音・環境音を自動で付けたい場合は、**Step 4B** を実行してから、ComfyUIの **「Workflow」→「開く」** で **`mmaudio_add_sound`** というワークフローを開いてください。

## 注意事項
- Google Colab の無料枠（T4 GPU / VRAM 16GB）での動作を想定し、既定では **fp8スケール版**（軽量・高品質）を使用します。VRAMが厳しい場合はモデルダウンロードのセルで **GGUF版** に切り替えられます。
- モデルファイルは合計 10GB前後になります。Googleドライブに保存しておくと、次回以降の再ダウンロードが不要になります（既定でON）。ComfyUI本体・カスタムノードは毎回ローカルディスクに再インストールされます（数分程度）。
- Colab は無操作が続くとランタイムが切断されます。長時間放置しないようにしてください。
- 生成した動画・画像はローカルにダウンロードするか、Googleドライブ保存先（`ComfyUI/output`）から取得してください。


---
## Step 0. GPUの確認


In [ ]:
gpu_check = get_ipython().getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
if not gpu_check or 'failed' in gpu_check[0].lower() or 'no devices' in gpu_check[0].lower():
    print('\u26a0\ufe0f GPUが有効になっていません。')
    print('上部メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU（T4など）を選択してから、')
    print('もう一度「ランタイム」→「すべてのセルを実行」してください。')
else:
    print('\u2705 GPU:', gpu_check[0])


---
## Step 1. 保存先の設定（Googleドライブ利用）
チェックを入れると、容量の大きい**モデルファイルだけ**Googleドライブに保存され、次回以降の再ダウンロードが不要になります（推奨）。

⚠️ ComfyUI本体・カスタムノード自体は、チェックの有無にかかわらず**常にColabのローカルディスク**にインストールされます
（セッション終了で消えますが数分で再インストールできます）。Googleドライブは細かいファイルの読み込みが非常に遅く、
ComfyUIのPythonファイルをそこに置くと起動だけで数分〜十数分かかることがあるためです。


In [ ]:
import os

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

# ComfyUI本体・カスタムノードは常にColabのローカルディスクに置く（高速・毎回再インストール）。
COMFYUI_DIR = '/content/ComfyUI'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # モデルファイル（数GB〜十数GB）だけはGoogleドライブに保存し、再ダウンロードを避ける。
    MODELS_BASE_DIR = '/content/drive/MyDrive/ComfyUI_Wan22_Models'
else:
    MODELS_BASE_DIR = '/content/ComfyUI_Wan22_Models'

os.makedirs(MODELS_BASE_DIR, exist_ok=True)
print('COMFYUI_DIR     =', COMFYUI_DIR)
print('MODELS_BASE_DIR =', MODELS_BASE_DIR)


---
## Step 2. ComfyUI 本体のインストール


In [ ]:
if not os.path.isdir(os.path.join(COMFYUI_DIR, '.git')):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFYUI_DIR}"
else:
    print('[skip] ComfyUI は既にインストール済みです。')

%cd {COMFYUI_DIR}
!pip install -q -r requirements.txt
print('\u2705 ComfyUI 本体のインストールが完了しました。')


---
## Step 3. ComfyUI-Manager と Wan2.2 関連カスタムノードのインストール


In [ ]:
CUSTOM_NODES_DIR = os.path.join(COMFYUI_DIR, 'custom_nodes')
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

def clone_node(url, name=None):
    name = name or url.rstrip('/').split('/')[-1].replace('.git', '')
    dest = os.path.join(CUSTOM_NODES_DIR, name)
    if os.path.isdir(os.path.join(dest, '.git')):
        print(f'[skip] {name} は既にインストール済みです。')
        return
    get_ipython().system(f'git clone --depth 1 {url} "{dest}"')
    req = os.path.join(dest, 'requirements.txt')
    if os.path.isfile(req):
        get_ipython().system(f'pip install -q -r "{req}"')

# ComfyUI管理・拡張機能マネージャー
clone_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
# Wan2.2 / WanVideo 系ノード（Kijai版・高度なワークフロー用）
clone_node('https://github.com/kijai/ComfyUI-WanVideoWrapper.git')
clone_node('https://github.com/kijai/ComfyUI-KJNodes.git')
# GGUF量子化モデルを読み込むためのノード
clone_node('https://github.com/city96/ComfyUI-GGUF.git')
# フレーム列を動画ファイルに書き出すためのノード
clone_node('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')
# （オプション）生成した動画に効果音・環境音を自動生成して合成するノード
clone_node('https://github.com/kijai/ComfyUI-MMAudio.git')

print('\u2705 カスタムノードのインストールが完了しました。')


---
## Step 4. Wan2.2 (I2V) モデルのダウンロード
- **fp8（推奨）**: 画質を保ったまま軽量化された版。T4 GPU（VRAM16GB）でも動作します。ComfyUI公式テンプレートと同じモデルです。
- **GGUF**: さらに軽量化したい場合の量子化版。量子化レベル（Q3〜Q8）を選べます。数値が小さいほど軽量・低画質になります。
  - GGUF版の配布リポジトリは頻繁に変わるため、`GGUF_REPO` は既定値で失敗する場合は
    [huggingface.co で `Wan2.2-I2V-A14B-GGUF` を検索](https://huggingface.co/models?search=Wan2.2-I2V-A14B-GGUF)して
    見つかったリポジトリID（例: `ユーザー名/リポジトリ名`）に書き換えてください。

モデルファイルは Hugging Face 上のファイル名を検索して取得するため、多少のファイル名変更があっても動作するようになっています。


In [ ]:
!pip install -q -U huggingface_hub

MODEL_SOURCE = "fp8 (\u63a8\u5968\u30fb\u9ad8\u54c1\u8cea)"  #@param ["fp8 (\u63a8\u5968\u30fb\u9ad8\u54c1\u8cea)", "gguf (\u7701VRAM)"]
GGUF_QUANT = "Q4_K_M"  #@param ["Q3_K_M", "Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
GGUF_REPO = "QuantStack/Wan2.2-I2V-A14B-GGUF"  #@param {type:"string"}


In [ ]:
import shutil
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()

MODELS_DIR = os.path.join(COMFYUI_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

def link_model_subdir(name):
    """モデルの実体は MODELS_BASE_DIR（Googleドライブ等）に置き、
    ComfyUI/models/<name> はそこへのシンボリックリンクにする。
    こうすることでComfyUI本体はローカルディスクの高速さを保ったまま、
    容量の大きいモデルファイルだけをドライブに永続化できる。"""
    target = os.path.join(MODELS_BASE_DIR, name)
    os.makedirs(target, exist_ok=True)
    link_path = os.path.join(MODELS_DIR, name)
    if not os.path.islink(link_path):
        if os.path.isdir(link_path):
            shutil.rmtree(link_path)
        elif os.path.exists(link_path):
            os.remove(link_path)
        os.symlink(target, link_path)
    return target

DIFFUSION_DIR = link_model_subdir('diffusion_models')
VAE_DIR = link_model_subdir('vae')
TEXT_ENC_DIR = link_model_subdir('text_encoders')
LORA_DIR = link_model_subdir('loras')

def find_and_download(repo_id, must_contain, dest_dir, exts=('.safetensors', '.gguf')):
    """repo_id 内で条件に合うファイルを検索し、dest_dir に無ければダウンロードする。
    （must_contain にはリポジトリ内パスの一部であるフォルダ名も含めてよい。
    実際に存在するかどうかは、検索で確定したファイル名そのもので判定する）"""
    files = api.list_repo_files(repo_id)
    matches = sorted(f for f in files if f.endswith(exts) and all(s.lower() in f.lower() for s in must_contain))
    if not matches:
        raise FileNotFoundError(
            f'{repo_id} 内に条件 {must_contain} に一致するファイルが見つかりませんでした。\n'
            f'リポジトリ内のファイル例: {files[:15]}\n'
            f'（GGUF_QUANT の値を変えるか、リポジトリIDを確認してください）'
        )
    filename = matches[0]
    flat_path = os.path.join(dest_dir, os.path.basename(filename))

    if os.path.isfile(flat_path):
        print(f'[skip] {os.path.basename(filename)} は既に存在します。')
        return

    print(f'Downloading: {repo_id}/{filename}')
    path = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dest_dir)

    if os.path.abspath(path) != os.path.abspath(flat_path):
        shutil.move(path, flat_path)
        nested_root = os.path.join(dest_dir, filename.split('/')[0])
        if os.path.isdir(nested_root):
            shutil.rmtree(nested_root, ignore_errors=True)
    print(f'  -> {flat_path}')

# ComfyUI公式ワークフローテンプレート（video_wan2_2_14B_i2v）で使われているものと同じリポジトリ
DIFFUSION_VAE_REPO = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
TEXT_ENCODER_REPO = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'

# 'i2v' を必ず条件に含める（t2v / fun_control / fun_camera など同じ high_noise/low_noise
# 命名を持つ他タスク向けファイルと誤って一致しないようにするため）
if MODEL_SOURCE.startswith('fp8'):
    find_and_download(DIFFUSION_VAE_REPO, ['diffusion_models', 'i2v', 'high_noise', 'fp8_scaled'], DIFFUSION_DIR)
    find_and_download(DIFFUSION_VAE_REPO, ['diffusion_models', 'i2v', 'low_noise', 'fp8_scaled'], DIFFUSION_DIR)
else:
    find_and_download(GGUF_REPO, ['i2v', 'high_noise', GGUF_QUANT.lower()], DIFFUSION_DIR)
    find_and_download(GGUF_REPO, ['i2v', 'low_noise', GGUF_QUANT.lower()], DIFFUSION_DIR)

# VAE / テキストエンコーダー
# 'wan_2.1' を条件に含める（同じ repo にある 5B モデル用 wan2.2_vae.safetensors と
# 誤って一致しないようにするため。14B I2V は wan_2.1_vae.safetensors を使う）
find_and_download(DIFFUSION_VAE_REPO, ['vae', 'wan_2.1'], VAE_DIR)
# 'fp8' を条件に含める（同じ repo にある非量子化版 umt5_xxl_fp16.safetensors（11GB超）と
# 誤って一致しないようにするため）
find_and_download(TEXT_ENCODER_REPO, ['text_encoders', 'umt5', 'fp8'], TEXT_ENC_DIR)

# 高速化用LoRA（ComfyUI公式のWan2.2 I2Vテンプレートが既定で参照するため必須）
find_and_download(DIFFUSION_VAE_REPO, ['loras', 'i2v', 'lightx2v', 'high_noise'], LORA_DIR)
find_and_download(DIFFUSION_VAE_REPO, ['loras', 'i2v', 'lightx2v', 'low_noise'], LORA_DIR)

print('\n\u2705 すべてのモデルのダウンロードが完了しました。')


---
## Step 4B. （オプション）動画に音声を自動生成して合成する（MMAudio）
生成した動画には音声が入っていません（Wan2.2は映像のみを生成するモデルです）。
ここでは [MMAudio](https://github.com/hkchengrex/MMAudio) というモデルを使い、動画の内容に合わせて
効果音・環境音を自動生成して合成します。音声合成が不要な場合はこのセルは実行しなくても問題ありません。


In [ ]:
import base64

MMAUDIO_DIR = link_model_subdir('mmaudio')
MMAUDIO_REPO = 'Kijai/MMAudio_safetensors'

find_and_download(MMAUDIO_REPO, ['mmaudio_large_44k_v2_fp16'], MMAUDIO_DIR)
find_and_download(MMAUDIO_REPO, ['mmaudio_vae_44k_fp16'], MMAUDIO_DIR)
find_and_download(MMAUDIO_REPO, ['mmaudio_synchformer_fp16'], MMAUDIO_DIR)
find_and_download(MMAUDIO_REPO, ['apple_dfn5b-clip-vit-h-14-384_fp16'], MMAUDIO_DIR)

# ComfyUIの「Workflow」メニューから開けるよう、サンプルワークフローを配置しておく
# （JSONはPythonソース中でクォート文字が問題にならないようbase64で埋め込んでいる）
WORKFLOWS_DIR = os.path.join(COMFYUI_DIR, 'user', 'default', 'workflows')
os.makedirs(WORKFLOWS_DIR, exist_ok=True)
MMAUDIO_WORKFLOW_B64 = 'eyJsYXN0X25vZGVfaWQiOiAxMDYsICJsYXN0X2xpbmtfaWQiOiAxNzYsICJub2RlcyI6IFt7ImlkIjogOTIsICJ0eXBlIjogIk1NQXVkaW9TYW1wbGVyIiwgInBvcyI6IFsyMzIyLjY0MTg0NTcwMzEyNSwgLTIyMi42NDAxNTE5Nzc1MzkwNl0sICJzaXplIjogWzQxMS4yOTMzOTU5OTYwOTM3NSwgMzU0XSwgImZsYWdzIjoge30sICJvcmRlciI6IDQsICJtb2RlIjogMCwgImlucHV0cyI6IFt7Im5hbWUiOiAibW1hdWRpb19tb2RlbCIsICJ0eXBlIjogIk1NQVVESU9fTU9ERUwiLCAibGluayI6IDE1Mn0sIHsibmFtZSI6ICJmZWF0dXJlX3V0aWxzIiwgInR5cGUiOiAiTU1BVURJT19GRUFUVVJFVVRJTFMiLCAibGluayI6IDE2Nn0sIHsibmFtZSI6ICJpbWFnZXMiLCAidHlwZSI6ICJJTUFHRSIsICJsaW5rIjogMTcwLCAic2hhcGUiOiA3fSwgeyJuYW1lIjogImR1cmF0aW9uIiwgInR5cGUiOiAiRkxPQVQiLCAibGluayI6IDE3NSwgIndpZGdldCI6IHsibmFtZSI6ICJkdXJhdGlvbiJ9fV0sICJvdXRwdXRzIjogW3sibmFtZSI6ICJhdWRpbyIsICJ0eXBlIjogIkFVRElPIiwgImxpbmtzIjogWzE2MywgMTcxXSwgInNsb3RfaW5kZXgiOiAwfV0sICJwcm9wZXJ0aWVzIjogeyJOb2RlIG5hbWUgZm9yIFMmUiI6ICJNTUF1ZGlvU2FtcGxlciJ9LCAid2lkZ2V0c192YWx1ZXMiOiBbMywgMjUsIDQuNSwgNywgImZpeGVkIiwgIiIsICIiLCBmYWxzZSwgdHJ1ZV19LCB7ImlkIjogOTgsICJ0eXBlIjogIlByZXZpZXdBdWRpbyIsICJwb3MiOiBbMjgzNS40MDA4Nzg5MDYyNSwgLTM1MS44MzA3NDk1MTE3MTg3NV0sICJzaXplIjogWzMxNSwgNzZdLCAiZmxhZ3MiOiB7fSwgIm9yZGVyIjogNSwgIm1vZGUiOiAwLCAiaW5wdXRzIjogW3sibmFtZSI6ICJhdWRpbyIsICJ0eXBlIjogIkFVRElPIiwgImxpbmsiOiAxNjN9XSwgIm91dHB1dHMiOiBbXSwgInByb3BlcnRpZXMiOiB7Ik5vZGUgbmFtZSBmb3IgUyZSIjogIlByZXZpZXdBdWRpbyJ9LCAid2lkZ2V0c192YWx1ZXMiOiBbbnVsbF19LCB7ImlkIjogOTcsICJ0eXBlIjogIlZIU19WaWRlb0NvbWJpbmUiLCAicG9zIjogWzI4MDUuNzU5Mjc3MzQzNzUsIC0xNjkuMzcyMzc1NDg4MjgxMjVdLCAic2l6ZSI6IFs2NDguNzQ1NDIyMzYzMjgxMiwgMTMyNS45OTI2NzU3ODEyNV0sICJmbGFncyI6IHt9LCAib3JkZXIiOiA2LCAibW9kZSI6IDAsICJpbnB1dHMiOiBbeyJuYW1lIjogImltYWdlcyIsICJ0eXBlIjogIklNQUdFIiwgImxpbmsiOiAxNjF9LCB7Im5hbWUiOiAiYXVkaW8iLCAidHlwZSI6ICJBVURJTyIsICJsaW5rIjogMTcxLCAic2hhcGUiOiA3fSwgeyJuYW1lIjogIm1ldGFfYmF0Y2giLCAidHlwZSI6ICJWSFNfQmF0Y2hNYW5hZ2VyIiwgImxpbmsiOiBudWxsLCAic2hhcGUiOiA3fSwgeyJuYW1lIjogInZhZSIsICJ0eXBlIjogIlZBRSIsICJsaW5rIjogbnVsbCwgInNoYXBlIjogN30sIHsibmFtZSI6ICJmcmFtZV9yYXRlIiwgInR5cGUiOiAiRkxPQVQiLCAibGluayI6IDE3NiwgIndpZGdldCI6IHsibmFtZSI6ICJmcmFtZV9yYXRlIn19XSwgIm91dHB1dHMiOiBbeyJuYW1lIjogIkZpbGVuYW1lcyIsICJ0eXBlIjogIlZIU19GSUxFTkFNRVMiLCAibGlua3MiOiBudWxsfV0sICJwcm9wZXJ0aWVzIjogeyJOb2RlIG5hbWUgZm9yIFMmUiI6ICJWSFNfVmlkZW9Db21iaW5lIn0sICJ3aWRnZXRzX3ZhbHVlcyI6IHsiZnJhbWVfcmF0ZSI6IDI0LCAibG9vcF9jb3VudCI6IDAsICJmaWxlbmFtZV9wcmVmaXgiOiAiTU1hdWRpbyIsICJmb3JtYXQiOiAidmlkZW8vaDI2NC1tcDQiLCAicGl4X2ZtdCI6ICJ5dXY0MjBwIiwgImNyZiI6IDE5LCAic2F2ZV9tZXRhZGF0YSI6IHRydWUsICJwaW5ncG9uZyI6IGZhbHNlLCAic2F2ZV9vdXRwdXQiOiBmYWxzZSwgInZpZGVvcHJldmlldyI6IHsiaGlkZGVuIjogZmFsc2UsICJwYXVzZWQiOiBmYWxzZSwgInBhcmFtcyI6IHsiZmlsZW5hbWUiOiAiTU1hdWRpb18wMDAxOS1hdWRpby5tcDQiLCAic3ViZm9sZGVyIjogIiIsICJ0eXBlIjogInRlbXAiLCAiZm9ybWF0IjogInZpZGVvL2gyNjQtbXA0IiwgImZyYW1lX3JhdGUiOiAxNn0sICJtdXRlZCI6IGZhbHNlfX19LCB7ImlkIjogMTAyLCAidHlwZSI6ICJNTUF1ZGlvRmVhdHVyZVV0aWxzTG9hZGVyIiwgInBvcyI6IFsxNTc3LjIxODg3MjA3MDMxMjUsIC02OC44NTQ5NDIzMjE3NzczNF0sICJzaXplIjogWzQ4Ny4yMzMyNzYzNjcxODc1LCAxNTRdLCAiZmxhZ3MiOiB7fSwgIm9yZGVyIjogMCwgIm1vZGUiOiAwLCAiaW5wdXRzIjogW3sibmFtZSI6ICJiaWd2Z2FuX3ZvY29kZXJfbW9kZWwiLCAidHlwZSI6ICJWT0NPREVSX01PREVMIiwgImxpbmsiOiBudWxsLCAic2hhcGUiOiA3fV0sICJvdXRwdXRzIjogW3sibmFtZSI6ICJtbWF1ZGlvX2ZlYXR1cmV1dGlscyIsICJ0eXBlIjogIk1NQVVESU9fRkVBVFVSRVVUSUxTIiwgImxpbmtzIjogWzE2Nl19XSwgInByb3BlcnRpZXMiOiB7Ik5vZGUgbmFtZSBmb3IgUyZSIjogIk1NQXVkaW9GZWF0dXJlVXRpbHNMb2FkZXIifSwgIndpZGdldHNfdmFsdWVzIjogWyJtbWF1ZGlvX3ZhZV80NGtfZnAxNi5zYWZldGVuc29ycyIsICJtbWF1ZGlvX3N5bmNoZm9ybWVyX2ZwMTYuc2FmZXRlbnNvcnMiLCAiYXBwbGVfREZONUItQ0xJUC1WaVQtSC0xNC0zODRfZnAxNi5zYWZldGVuc29ycyIsICI0NGsiLCAiZnAxNiJdfSwgeyJpZCI6IDg1LCAidHlwZSI6ICJNTUF1ZGlvTW9kZWxMb2FkZXIiLCAicG9zIjogWzE2NDIuNjEzODkxNjAxNTYyNSwgLTIyOC4zOTM3MDcyNzUzOTA2Ml0sICJzaXplIjogWzQzOC4yMTk5NzA3MDMxMjUsIDgzLjg3OTk4OTYyNDAyMzQ0XSwgImZsYWdzIjoge30sICJvcmRlciI6IDEsICJtb2RlIjogMCwgImlucHV0cyI6IFtdLCAib3V0cHV0cyI6IFt7Im5hbWUiOiAibW1hdWRpb19tb2RlbCIsICJ0eXBlIjogIk1NQVVESU9fTU9ERUwiLCAibGlua3MiOiBbMTUyXSwgInNsb3RfaW5kZXgiOiAwfV0sICJwcm9wZXJ0aWVzIjogeyJOb2RlIG5hbWUgZm9yIFMmUiI6ICJNTUF1ZGlvTW9kZWxMb2FkZXIifSwgIndpZGdldHNfdmFsdWVzIjogWyJtbWF1ZGlvX2xhcmdlXzQ0a192Ml9mcDE2LnNhZmV0ZW5zb3JzIiwgImZwMTYiXX0sIHsiaWQiOiAxMDUsICJ0eXBlIjogIlZIU19WaWRlb0luZm8iLCAicG9zIjogWzIwMjguMzA0NTY1NDI5Njg3NSwgMjM2LjE1Mjg3NzgwNzYxNzJdLCAic2l6ZSI6IFsyNjIsIDIwNl0sICJmbGFncyI6IHt9LCAib3JkZXIiOiAzLCAibW9kZSI6IDAsICJpbnB1dHMiOiBbeyJuYW1lIjogInZpZGVvX2luZm8iLCAidHlwZSI6ICJWSFNfVklERU9JTkZPIiwgImxpbmsiOiAxNzJ9XSwgIm91dHB1dHMiOiBbeyJuYW1lIjogInNvdXJjZV9mcHPwn5+oIiwgInR5cGUiOiAiRkxPQVQiLCAibGlua3MiOiBudWxsfSwgeyJuYW1lIjogInNvdXJjZV9mcmFtZV9jb3VudPCfn6giLCAidHlwZSI6ICJJTlQiLCAibGlua3MiOiBudWxsfSwgeyJuYW1lIjogInNvdXJjZV9kdXJhdGlvbvCfn6giLCAidHlwZSI6ICJGTE9BVCIsICJsaW5rcyI6IG51bGx9LCB7Im5hbWUiOiAic291cmNlX3dpZHRo8J+fqCIsICJ0eXBlIjogIklOVCIsICJsaW5rcyI6IG51bGx9LCB7Im5hbWUiOiAic291cmNlX2hlaWdodPCfn6giLCAidHlwZSI6ICJJTlQiLCAibGlua3MiOiBudWxsfSwgeyJuYW1lIjogImxvYWRlZF9mcHPwn5+mIiwgInR5cGUiOiAiRkxPQVQiLCAibGlua3MiOiBbMTc2XSwgInNsb3RfaW5kZXgiOiA1fSwgeyJuYW1lIjogImxvYWRlZF9mcmFtZV9jb3VudPCfn6YiLCAidHlwZSI6ICJJTlQiLCAibGlua3MiOiBudWxsLCAic2xvdF9pbmRleCI6IDZ9LCB7Im5hbWUiOiAibG9hZGVkX2R1cmF0aW9u8J+fpiIsICJ0eXBlIjogIkZMT0FUIiwgImxpbmtzIjogWzE3NV0sICJzbG90X2luZGV4IjogN30sIHsibmFtZSI6ICJsb2FkZWRfd2lkdGjwn5+mIiwgInR5cGUiOiAiSU5UIiwgImxpbmtzIjogbnVsbH0sIHsibmFtZSI6ICJsb2FkZWRfaGVpZ2h08J+fpiIsICJ0eXBlIjogIklOVCIsICJsaW5rcyI6IG51bGx9XSwgInByb3BlcnRpZXMiOiB7Ik5vZGUgbmFtZSBmb3IgUyZSIjogIlZIU19WaWRlb0luZm8ifSwgIndpZGdldHNfdmFsdWVzIjoge319LCB7ImlkIjogOTEsICJ0eXBlIjogIlZIU19Mb2FkVmlkZW8iLCAicG9zIjogWzE1NTkuOTEyNDc1NTg1OTM3NSwgMTc0LjcwMjE3ODk1NTA3ODEyXSwgInNpemUiOiBbNDE3LjY2MTc3MzY4MTY0MDYsIDkxMi4yNTg4NTAwOTc2NTYyXSwgImZsYWdzIjoge30sICJvcmRlciI6IDIsICJtb2RlIjogMCwgImlucHV0cyI6IFt7Im5hbWUiOiAibWV0YV9iYXRjaCIsICJ0eXBlIjogIlZIU19CYXRjaE1hbmFnZXIiLCAibGluayI6IG51bGwsICJzaGFwZSI6IDd9LCB7Im5hbWUiOiAidmFlIiwgInR5cGUiOiAiVkFFIiwgImxpbmsiOiBudWxsLCAic2hhcGUiOiA3fV0sICJvdXRwdXRzIjogW3sibmFtZSI6ICJJTUFHRSIsICJ0eXBlIjogIklNQUdFIiwgImxpbmtzIjogWzE2MSwgMTcwXSwgInNsb3RfaW5kZXgiOiAwfSwgeyJuYW1lIjogImZyYW1lX2NvdW50IiwgInR5cGUiOiAiSU5UIiwgImxpbmtzIjogbnVsbH0sIHsibmFtZSI6ICJhdWRpbyIsICJ0eXBlIjogIkFVRElPIiwgImxpbmtzIjogbnVsbH0sIHsibmFtZSI6ICJ2aWRlb19pbmZvIiwgInR5cGUiOiAiVkhTX1ZJREVPSU5GTyIsICJsaW5rcyI6IFsxNzJdLCAic2xvdF9pbmRleCI6IDN9XSwgInByb3BlcnRpZXMiOiB7Ik5vZGUgbmFtZSBmb3IgUyZSIjogIlZIU19Mb2FkVmlkZW8ifSwgIndpZGdldHNfdmFsdWVzIjogeyJ2aWRlbyI6ICJDb2dWaWRlb1hfMV81X0kyVl8wMDA5Mi5tcDQiLCAiZm9yY2VfcmF0ZSI6IDAsICJmb3JjZV9zaXplIjogIkRpc2FibGVkIiwgImN1c3RvbV93aWR0aCI6IDUxMiwgImN1c3RvbV9oZWlnaHQiOiA1MTIsICJmcmFtZV9sb2FkX2NhcCI6IDAsICJza2lwX2ZpcnN0X2ZyYW1lcyI6IDAsICJzZWxlY3RfZXZlcnlfbnRoIjogMSwgImNob29zZSB2aWRlbyB0byB1cGxvYWQiOiAiaW1hZ2UiLCAidmlkZW9wcmV2aWV3IjogeyJoaWRkZW4iOiBmYWxzZSwgInBhdXNlZCI6IGZhbHNlLCAicGFyYW1zIjogeyJmcmFtZV9sb2FkX2NhcCI6IDAsICJza2lwX2ZpcnN0X2ZyYW1lcyI6IDAsICJmb3JjZV9yYXRlIjogMCwgImZpbGVuYW1lIjogIkNvZ1ZpZGVvWF8xXzVfSTJWXzAwMDkyLm1wNCIsICJ0eXBlIjogImlucHV0IiwgImZvcm1hdCI6ICJ2aWRlby9tcDQiLCAic2VsZWN0X2V2ZXJ5X250aCI6IDF9LCAibXV0ZWQiOiBmYWxzZX19fV0sICJsaW5rcyI6IFtbMTUyLCA4NSwgMCwgOTIsIDAsICJNTUFVRElPX01PREVMIl0sIFsxNjEsIDkxLCAwLCA5NywgMCwgIklNQUdFIl0sIFsxNjMsIDkyLCAwLCA5OCwgMCwgIkFVRElPIl0sIFsxNjYsIDEwMiwgMCwgOTIsIDEsICJNTUFVRElPX0ZFQVRVUkVVVElMUyJdLCBbMTcwLCA5MSwgMCwgOTIsIDIsICJJTUFHRSJdLCBbMTcxLCA5MiwgMCwgOTcsIDEsICJBVURJTyJdLCBbMTcyLCA5MSwgMywgMTA1LCAwLCAiVkhTX1ZJREVPSU5GTyJdLCBbMTc1LCAxMDUsIDcsIDkyLCAzLCAiRkxPQVQiXSwgWzE3NiwgMTA1LCA1LCA5NywgNCwgIkZMT0FUIl1dLCAiZ3JvdXBzIjogW10sICJjb25maWciOiB7fSwgImV4dHJhIjogeyJkcyI6IHsic2NhbGUiOiAwLjY4MzAxMzQ1NTM2NTA3MDYsICJvZmZzZXQiOiBbLTEwNjEuNTkzNzY4NTE4NzQ0LCA0NTEuMDg1MDkyOTYyNDc3M119fSwgInZlcnNpb24iOiAwLjR9'
MMAUDIO_WORKFLOW_JSON = base64.b64decode(MMAUDIO_WORKFLOW_B64).decode('utf-8')
with open(os.path.join(WORKFLOWS_DIR, 'mmaudio_add_sound.json'), 'w') as f:
    f.write(MMAUDIO_WORKFLOW_JSON)

print('\u2705 MMAudio のセットアップが完了しました。')
print('ComfyUIの「Workflow」→「開く」から \'mmaudio_add_sound\' を開いて使ってください。')


---
## Step 4C. （オプション）Wan2.2-S2V（音声駆動ビデオ生成）の導入
[Wan2.2-S2V](https://huggingface.co/Wan-AI/Wan2.2-S2V-14B) は、**1枚の画像 + 音声ファイル**から、
その音声に合わせて自然に動く（リップシンクする）動画を生成するモデルです。
ComfyUI本体に標準で対応しているため、追加のカスタムノードは不要で、モデルのダウンロードだけで使えます。
不要な場合はこのセルを実行しなくても問題ありません。


In [ ]:
AUDIO_ENCODER_DIR = link_model_subdir('audio_encoders')

# S2V は high_noise/low_noise の2モデル構成ではなく、単一の拡散モデルを使う
# 's2v' を必ず条件に含める（i2v/t2v向けファイルと誤って一致しないようにするため）
find_and_download(DIFFUSION_VAE_REPO, ['diffusion_models', 's2v', 'fp8_scaled'], DIFFUSION_DIR)
find_and_download(DIFFUSION_VAE_REPO, ['audio_encoders', 'wav2vec2'], AUDIO_ENCODER_DIR)
# VAE / テキストエンコーダーは Step 4 でダウンロード済みのものをそのまま使う

print('\u2705 Wan2.2-S2V のモデルダウンロードが完了しました。')
print('ComfyUIの「Workflow」→「Browse Templates」→「Video」から')
print('「Wan2.2-S2V音声駆動型ビデオ生成」を開いて使ってください。')


---
## Step 5. Cloudflare Tunnel (cloudflared) の準備


In [ ]:
CLOUDFLARED_PATH = '/content/cloudflared'

if not os.path.isfile(CLOUDFLARED_PATH):
    !wget -q -O {CLOUDFLARED_PATH} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CLOUDFLARED_PATH}
    print('\u2705 cloudflared をインストールしました。')
else:
    print('[skip] cloudflared は既にインストール済みです。')


---
## Step 6. ComfyUI の起動 ＋ 公開URLの発行
実行すると ComfyUI がバックグラウンドで起動し、Cloudflare Tunnel の公開URLが表示されます。
URLが表示されたら、そのリンクをクリックしてブラウザで開いてください（初回はモデル読み込みで数十秒かかることがあります）。


In [ ]:
import subprocess, sys, time, re

%cd {COMFYUI_DIR}

COMFYUI_PORT = 8188

# start_new_session=True: このセルの実行が終わったり、Colabのカーネルに
# 割り込み(SIGINT/SIGTERM)が送られたりしても、バックグラウンドプロセスが
# 巻き添えで終了しないよう、ノートブックのプロセスグループから切り離す。
# --enable-cors-header: ComfyUIには「別オリジンのページからのアクセスを
# 拒否する」保護機能(DNS rebinding対策)が入っており、Colabのページから
# cloudflaredのURLへリンク遷移すると 'cross-site' 扱いになって403 Forbidden
# になってしまう。このフラグでその保護を無効化し、トンネル経由のアクセスを許可する。
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    [sys.executable, '-u', 'main.py', '--listen', '0.0.0.0', '--port', str(COMFYUI_PORT),
     '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT, cwd=COMFYUI_DIR,
    start_new_session=True,
)
print(f'\u25b6\ufe0f ComfyUI をバックグラウンドで起動しました (PID: {comfy_proc.pid})')

tunnel_log_path = '/content/cloudflared.log'
tunnel_log = open(tunnel_log_path, 'w')
# stdbuf -oL でcloudflaredの標準出力を行単位フラッシュに強制する。
# ファイルにリダイレクトするとブロック単位バッファリングになり、
# URLが実際にログへ書き出されるまで時間がかかることがあるための対策。
# --protocol http2: 既定のQUIC(UDP)はColabのネットワーク環境で不安定になりやすく、
# 接続してもCloudflare側で 'Error 1033 (Tunnel error)' になることがあるため、
# より安定するHTTP/2を明示的に指定する。
tunnel_proc = subprocess.Popen(
    ['stdbuf', '-oL', '-eL', CLOUDFLARED_PATH, 'tunnel', '--url', f'http://127.0.0.1:{COMFYUI_PORT}',
     '--protocol', 'http2', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f'\u25b6\ufe0f Cloudflare Tunnel をバックグラウンドで起動しました (PID: {tunnel_proc.pid})')

print('\n\u23f3 公開URLの発行を待っています...')

url = None
tunnel_crashed = False
MAX_WAIT_SEC = 360
POLL_SEC = 2
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    with open(tunnel_log_path) as f:
        content = f.read()
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', content)
    if m:
        url = m.group(0)
        break
    # cloudflared が起動直後に落ちていないか確認する。
    # （不正なフラグ等で即終了した場合、そのまま待ち続けても永遠にURLは出ない）
    if tunnel_proc.poll() is not None:
        tunnel_crashed = True
        break
    elapsed = (i + 1) * POLL_SEC
    if elapsed % 30 == 0:
        print(f'  ...まだ待機中です（経過 {elapsed}秒）。トンネル確立まで数分かかる場合があります。')

if tunnel_crashed:
    print('\n\u26a0\ufe0f cloudflared が起動直後に終了しました。以下はログの内容です:\n')
    with open(tunnel_log_path) as f:
        print(f.read())
# URLが発行された直後はCloudflareのエッジへの接続登録がまだ完了しておらず、
# すぐ開くと 'Error 1033 (Tunnel error)' になることがあるため、
# 'Registered tunnel connection'（接続確立完了）のログが出るまで少し待つ。
# ログの文言が変わっていても無限待機しないよう、最大15秒で打ち切って先に進む。
if url:
    for _ in range(15):
        with open(tunnel_log_path) as f:
            content = f.read()
        if 'Registered tunnel connection' in content:
            break
        time.sleep(1)

if url:
    print('\n' + '=' * 64)
    print('\u2705 下のURLをクリックして ComfyUI を開いてください:')
    print()
    print(url)
    print()
    print('（Error 1033 や真っ白な画面が出た場合は、接続確立中の可能性があります。')
    print('  10〜30秒ほど待ってから再読み込みしてください）')
    print('=' * 64)
elif not tunnel_crashed:
    print('\u26a0\ufe0f Tunnel URL の取得にタイムアウトしました。下のセルで /content/cloudflared.log を確認してください。')


---
### （トラブル時）ログの確認用セル
ComfyUIが起動しない・エラーが出る場合はこのセルを実行してログを確認してください。


In [ ]:
print('--- ComfyUI ログ（末尾40行） ---')
!tail -n 40 /content/comfyui.log
print()
print('--- cloudflared ログ（末尾20行） ---')
!tail -n 20 /content/cloudflared.log


---
### （終了時）プロセス停止用セル
作業が終わったら、下のチェックボックスをONにしてからこのセルを実行してください（Colabのランタイムを切断すれば自動的に停止するので、このセルは必須ではありません）。

⚠️ **「すべてのセルを実行」をした時にこのセルも一緒に実行され、
起動したばかりのComfyUIを誤って停止してしまわないよう、既定ではチェックボックスがOFF（何もしない）になっています。**


In [ ]:
CONFIRM_STOP = False  #@param {type:"boolean"}

if not CONFIRM_STOP:
    print('チェックボックスがOFFのため、何もしませんでした。')
    print('本当に停止したい場合は、上のチェックボックスをONにしてからこのセルだけを実行してください。')
else:
    for p, name in [(globals().get('comfy_proc'), 'ComfyUI'), (globals().get('tunnel_proc'), 'cloudflared')]:
        if p is not None:
            p.terminate()
            print(f'{name} (PID: {p.pid}) を停止しました。')
        else:
            print(f'{name} のプロセスは見つかりませんでした。')
